# धडा 18: क्रिप्टोग्राफिक रिसीटसह AI एजंट्सचे संरक्षण करणे

## प्रत्यक्ष नोटबुक

हा नोटबुक चार क्रियांमधून मार्गदर्शन करतो:

1. **तुमचा पहिला रिसीट साइन करा** एजंट टूल कॉल साठी आणि त्याची पडताळणी करा.
2. **रिसीट मध्ये फेरफार करा** आणि पडताळणी अयशस्वी होते ते पाहा.
3. **तीन रिसीटचा साखळ तयार करा** आणि साखळीची अखंडता पुष्टी करा.
4. **मायक्रोसॉफ्ट एजंट फ्रेमवर्क टूल कॉलला व्रॅप करा** त्यामुळे प्रत्येक क्रियेमुळे रिसीट निर्माण होईल.

सर्व क्रिप्टोग्राफिक प्रिमिटिव्हज चांगल्या देखभाल केलेल्या लायब्ररींतून आयात केलेले आहेत (`pynacl` Ed25519 साठी, `jcs` RFC 8785 canonical JSON साठी, `hashlib` Python स्टँडर्ड लायब्ररीमधून SHA-256 साठी). रिसीट लॉजिक स्वतः साधा Python आहे ज्याला तुम्ही वाचू आणि सुधारू शकता.

सेल्स अनुक्रमे चालवा. प्रत्येक विभाग लहान आणि स्वतंत्र आहे.


## सेटअप

दोन अवलंबन स्थापित करा. दोन्ही परवानगी देणाऱ्या परवानग्यांसह आहेत (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## मदतनीस उपयुक्तता

हे दोन मदतनीस बेस64url एनकोडिंग (पॅडिंगशिवाय) आणि मनमानी ऑब्जेक्ट्सच्या SHA-256 हॅशिंगची हाताळणी करतात. ते नोटबुकचा उर्वरित भाग रिसिप्ट लॉजिकवर लक्ष केंद्रित ठेवतात.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## विभाग 1: आपला पहिला पावतीवर सही करा

कल्पना करा की **Contoso Travel** साठी आमचा एजंट नुकताच सिडनी ते लॉस एंजेलिस प्रवासासाठी ग्राहकासाठी फ्लाइट पाहत आहे. आम्हाला हे उपकरण कॉल एक सही केलेल्या पावती म्हणून नोंदवायचे आहे जेणेकरून भविष्यातील ऑडिटर आम्हांवर विश्वास न ठेवता याची पडताळणी करू शकेल.

### टप्पा 1.1: एक सही करणारी की तयार करा

उत्पादनात, एजंटची सही करणारी की हार्डवेअर सिक्युरिटी मॉड्यूल (HSM), Azure Key Vault, किंवा तत्सम संरक्षित साठवणीत ठेवली जात असते. या धड्यासाठी आम्ही मेमरीमध्ये नवीन की तयार करतो.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### टप्पा 1.2: रिसीट पेलोड तयार करा

पेलोडमध्ये ते सर्व काही असते जे आम्हाला रिसीटमध्ये प्रमाणित करायचे असते: कोण कार्यरत होते, कोणते साधन वापरले, कोणते अर्ग्युमेंट्स दिले, काय परत आले, कोणत्या धोरणांतर्गत, आणि केव्हा. रिसीटमध्ये संवेदनशील माहिती गुप्त ठेवण्यासाठी आम्ही अर्ग्युमेंट्स आणि निकाल यांचे हॅश करतो, त्यांना थेट समाविष्ट करत नाही.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### टप्पा 1.3: स्वाक्षरी करा आणि पावती एकत्र करा

तीन टप्पे:

1. JCS वापरून पेलोडचे सामान्यीकरण करा जेणेकरून समान तार्किक पावती तयार करणाऱ्या दोन अंमलबजावण्या बाइट-ओळखीचे बाइट तयार करतील.
2. सामान्यीकृत बाइट्सवर SHA-256 ने हॅश करा.
3. Ed25519 खाजगी कीने हॅशवर स्वाक्षरी करा.

नंतर स्वाक्षरी मूळ पेलोडशी संलग्न केली जाते जे अंतिम पावती तयार करते.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### टप्पा 1.4: पावतीची पडताळणी करा

पडताळणी प्रक्रियेला उलटते. आम्ही स्वाक्षरी काढतो, canonical hash पुन्हा मोजतो, आणि पावतीतील सार्वजनिक कीशी स्वाक्षरी तपासतो.

ही पडताळणी करणारा एक लेखापरीक्षक आम्हाला पावती व्यतिरिक्त काहीही गरज नसते. कोणतेही सेवा कॉल करायची गरज नाही, कोणत्याही की निर्देशिकेची चौकशी करायची गरज नाही, कोणत्याही विश्वासाची गरज नाही.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

तुम्हाला `Receipt is valid: True` दिसायला हवे. एजंटने त्याचा पहिला क्रिप्टोग्राफिकली स्वाक्षरी केलेला ऑडिट रेकॉर्ड तयार केला आहे.


## विभाग 2: पावतीशी छेडछाड करा

पावत्या म्हणजे त्या छेडछाड उघड करणार्‍या असतात हाच संपूर्ण उद्देश आहे. चला हे सिद्ध करूया.

आम्ही पावतीतील नक्कीच एकच अक्षर बदलू आणि पडताळणी अयशस्वी होताना पाहू.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### काय घडलं नुकतंच?

जेव्हा आपण `policy_id` बदलले, तेव्हा canonical bytes बदलले. त्या bytes चा SHA-256 hash बदलला. मूळ hash वर असलेली सही आता नवीन hash शी जुळत नाही. पडताळणी योग्यरित्या `False` परत करते.

पावतीतील कोणताही फील्ड बदलण्याचा आणि तरीही ती पडताळणी होण्याचा कोणताही मार्ग नाही, जोपर्यंत ह्याकरिता हल्लेखोराकडे खाजगी की नसेल. जोपर्यंत खाजगी की की वॉल्टमध्ये आहे आणि सार्वजनिक की प्रकाशित आहे, तोपर्यंत छेडछाड लपवणं अशक्य आहे.

स्वतः प्रयत्न करा: वरील सेलमध्ये `tool_name` किंवा `agent_id` किंवा `timestamp` बदला आणि पुन्हा चालवा. प्रत्येक बदल चुकीच्या पावतीचे उत्पादन करतो.


## विभाग 3: पावत्या एकत्र साखळी करा

एकच पावती एका क्रियेचे संरक्षण करते. बहुतांश एजंट अनेक क्रिया करतात. संपूर्ण क्रमानंतर छेडछाड-प्रतिबंधक करण्यासाठी, आपण नवीन पावतीच्या पेलोडमध्ये मागील पावतीचा हॅश समाविष्ट करून प्रत्येक पावतीला मागील पावतीशी लिंक करतो.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

जर कोणी पावती काढली किंवा पुनर्व्यवस्थित केली, तर साखळी त्या नेमक्या ठिकाणी तुटते. कोणत्याही नंतरच्या पावतीचे पडताळणी अयशस्वी होते कारण त्याचा `previous_receipt_hash` आता त्याच्या पूर्वीसारख्या हॅशशी जुळत नाही.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

आता साखळी तोडण्यासाठी मधल्या पावतीत फेरबदल करा आणि पुन्हा तपासा. फेरबदल केलेली पावती तिच्या सही तपासणीत अयशस्वी होते, आणि पुढील पावतीची साखळी-लिंक तपासणी देखील अयशस्वी होते (कारण तिचा `previous_receipt_hash` आता सुधारित केलेल्या मधल्या पावतीच्या हॅशशी जुळत नाही).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

पावती 0 अजूनही सत्यापित करते (ते सुधारित केलेले नाही आणि त्याचा अवलंब करायचा पूर्ववर्ती नाही). पावती 1 तिच्या स्वाक्षरी तपासणीमध्ये अपयशी ठरते कारण आपण `tool_args_hash` बदलले आहे. पावती 2 तिच्या साखळी-संपर्क तपासणीमध्ये अपयशी ठरते कारण तिचा `previous_receipt_hash` मूळ (आता सुधारित) पावती 1 विरुद्ध गणना केला गेला होता.

जरी एखादा हल्लेखोर सुधारित पावती 1 पुन्हा स्वाक्षरी करत असेल (जी ते खाजगी कळीशिवाय करू शकत नाहीत), तरीही पावती 2 मधील साखळी-संपर्क विसंगती फेरफार दाखवेल. बदल लपविण्यासाठी, हल्लेखोराला सुधारणेच्या बिंदूपासून पुढील प्रत्येक पावती पुन्हा स्वाक्षरी करावी लागेल, ज्यासाठी खाजगी कळीची मालकी आवश्यक आहे.


## विभाग ४: एजंट टूल कॉलला रसीद सहीसह वेढा घाला

वास्तविक तैनातीमध्ये, तुम्हाला प्रत्येक एजंट लेखकाला `make_receipt` कॉल करणे लक्षात ठेवण्याची गरज नाही. तुम्हाला प्रत्येक टूल कॉलसाठी रसीद सही आपोआप होणे पाहिजे.

येथे सर्वात सोपा नमुना आहे: एक वेप्त क्लास जो कोणत्याही कॉल करण्यायोग्य टूल फंक्शनला घेतो आणि त्याचा रसीद-निर्गमन करणारा आवृत्ती परत करतो. हे कोणत्याही एजंट फ्रेमवर्कशी जुळते, ज्यात Microsoft एजंट फ्रेमवर्क (`agent_framework.foundry`) देखील समाविष्ट आहे.

तुमच्याकडे Microsoft Foundry प्रोजेक्ट सेटअप नसल्यास, खालील स्थानिक मॉक अजूनही ह्या नमुन्याचे प्रदर्शन करतो.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### मायक्रोसॉफ्ट एजंट फ्रेमवर्कसह समाकलन

वरील `ReceiptedTool` व्रॅपर फ्रेमवर्क-स्वतंत्र आहे. मायक्रोसॉफ्ट एजंट फ्रेमवर्क वापरुन तयार केलेल्या एजंटमध्ये त्याचा वापर करण्यासाठी, व्रॅप केलेल्या फंक्शनला टूल म्हणून नोंदणी करा. एक आराखडा (आपण मॉकला खऱ्या मायक्रोसॉफ्ट फाउंड्री टूल नोंदणीने बदली कराल):

```python
# एकत्रीकरण आकार दर्शविणारे छद्मकोड.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="तुम्ही Contoso Travel एजंट आहात ...",
#     tools=[receipted_lookup],   # रॅप केलेले साधन, कच्च्या फंक्शन नाही
# )
# response = agent.run("सिडनीहून लॉस एंजेलिसपर्यंत जूनमध्ये फ्लाइट शोधा.")
#
# # रन नंतर, एजंटने केलेल्या प्रत्येक साधन कॉलला एक सही केलेली पावती आहे:
# audit_chain = receipted_lookup.receipts
```

एजंट फ्रेमवर्कला रसीदांविषयी काहीही जाणून घेण्याची गरज नाही. रसीद स्वाक्षरी टूलच्या भोवती व्रॅप केली जाते, फ्रेमवर्कमध्ये घातलेली नाही. एजंटला पुन्हा लिहिण्याशिवाय विद्यमान एजंट कोडमध्ये आपल्या कष्टाचा पुरावा कसा जोडायचा हे असे आहे.


## पुनरावलोकन आणि विस्तार आव्हान

तुमच्याकडे आहे:

- एका Ed25519 की जोडीची निर्मिती केली.
- एजंट टूल कॉलसाठी पावत तयार केली आणि सही केली.
- केवळ सार्वजनिक किल्लीचा वापर करून ऑफलाइन पावत सत्यापित केली.
- पावतीमध्ये फेरफार केला आणि सत्यापन फेल झाले पाहिले.
- तीन पावत्या असलेली हॅश-चेन केलेली शृंखला तयार केली.
- शृंखलेच्या मधोमध फेरफार केला आणि सहीचा तसेच साखळी-लिंकचा अपयश पाहिले.
- एक टूल फंक्शन ऑटोमॅटिक पावती सहीसह तयार केली.

**विस्तार आव्हान.** पावत स्कीमामध्ये `request_id` फील्ड (वितरित ट्रेसिंगसाठी UUID) जोडा. `make_receipt` ला अपडेट करा ज्याने तो समाविष्ट होईल, आणि पावत्या अखेरीस तपासल्या जातात हे पुष्टी करा. नंतर सही केल्यानंतर फील्ड बदलून तपासणी अयशस्वी होते हे पाहा. यामुळे तुम्हाला कॅनॉनिकल एन्कोडिंगच्या प्रत्येक बाइटचा सहीत कसा वाटा असतो हे आत्मसात करावे लागेल.

**महत्त्वाचा सीमा.** पावत्या तीन गोष्टींचा पुरावा करतात आणि फक्त तीच: वाटप (ही की या सामग्रीवर सही केली), अखंडता (सामग्री सही झाल्यापासून बदललेली नाही), आणि क्रमवारी (ही पावती त्या पावतीनंतर आली). त्या पुरावा देत नाही की एजंटची क्रिया बरोबर होती, `policy_id` मध्ये नाव दिलेला धोरण प्रत्यक्षात तपासला गेला, किंवा एजंटने प्रत्येक नियम पाळला. पावत्या एक पाया आहेत. शासन तुम्ही त्यावर बांधता.

हा धडा पुन्हा README वाचा आणि त्या सीमेकडे लक्ष द्या. पावत्यांसोबत टीम्सची सर्वात सामान्य चूक म्हणजे "आपल्याकडे पावत्या आहेत" म्हणजे "आपल्यावर शासन आहे" असं समजणे. तसे नाही. पावत्या एजंटच्या वर्तनाचा लेखाजोखा ठेवतात. त्या त्याला बरोबर ठरवत नाहीत.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
